A syd visualizer for heading direction vs head orientation in homings and escapes

In [1]:
%reload_ext autoreload
%autoreload 2

import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from syd import make_viewer

from behave_analysis.process.process import Process
from behave_analysis.visualize.visualize_utils import open_tracking_data
from behave_analysis.analyze.behaviour.homings_escapes.homings import get_Homings
from databank import full_experiments_objects as experiments_objects

settings = {"homings_speed_threshold": 4.0,  # cm/s, used to find bouts of running that may be homings
            "homings_gap_tolerance": 1,  # frames, used to merge bouts
            "homings_features_initial_window_s": 1.0,  # seconds, used to compute initial features of homings like acceleration and hdir change
            "homing_classification_target_recall": 0.9,  # minimum recall for a gate to be considered valid
            "homings_classification_recall_threshold": 0.9,  # minimum recall for a feature gate to be considered valid
            "homings_classification_precision_threshold": 0.1,  # minimum precision for a feature gate to be considered valid
            "homings_classification_auc_threshold": 0.9,  # or .8, minimum AUC for a feature gate to be considered valid
            "homings_classification_cohens_d_threshold": 1,  # minimum absolute Cohen's d for a feature gate to be considered valid
            "redo_compute": False,
            "homings_use_boris": False,
            "homings_curated": False,
            "homings_distance_threshold": 25  # in cm, minimum length to be kept as a homings
            }
from settings.settings_analyze_behave import settings_ab
from settings.settings_overrides import settings_overrides
settings_ab = settings_overrides(settings_ab, settings)

%matplotlib inline

In [2]:
# choose a session and load its data
e = 31

cond_list = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
exp = experiments_objects[e]
session = Process(exp).load_session()

tracking_data = open_tracking_data(session)
video_df_path = os.path.join(session.base_path, session.processed_path, 'full_video_dataframe.csv')
video_df = pl.read_csv(video_df_path)

# open database and check for run with matched settings - if it doesn't exist run it!
homings_dict = get_Homings(settings_ab, session).get_homings(video_df=[], tracking_data=[])  
print(f"Found {len(homings_dict['onset_frames'])} homings in this session")

2026-07-08 13:00:57.531 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings:__init__:41 - checking for existing homings results
2026-07-08 13:00:57.600 | INFO     | behave_analysis.analyze.results_database_utils:check_database_for_same_run:23 - Found 1 matched results in database: ['4a0b393b225c443b'] in the folder Z:\Jasmine_Laurence\Experimental_Data\JAL007\JAL007_barrierflip2_2024_03_12T11_18_26\processed_data\homings\Homing_database.csv
2026-07-08 13:00:57.602 | INFO     | behave_analysis.analyze.behaviour.homings_escapes.homings:get_homings:58 - Homing analysis already done with these settings, loading from database...


Found 108 homings in this session


In [27]:
from behave_analysis.utils.arena_plotting import Arena

# Parameters
speed_threshold = 4.0   # cm/s

viewer = make_viewer()
viewer.add_integer('idx', min=0, max=len(homings_dict["onset_frames"]) - 1, value=0)
viewer.add_integer('diff_window', min=1, max=10, value=3)
viewer.add_integer('arrow_every', min=1, max=10, value=3) # plot every Nth frame to reduce clutter
viewer.add_integer('arrow_len', min=1, max=50, value=20) # arrow length in position units (cm if your coords are cm)

def see_next(state):
    viewer.update_integer("idx", value=state["idx"] + 1)
viewer.add_button("see_next", label="See next", callback=see_next)

def plot(state):
    # Homing segment
    start = homings_dict["onset_frames"][state["idx"]]-state["diff_window"]
    end = homings_dict["offset_frames"][state["idx"]]

    x = np.asarray(tracking_data["body_loc"][start:end, 0], dtype=float)
    y = np.asarray(tracking_data["body_loc"][start:end, 1], dtype=float)

    # Displacement over diff_window frames (NumPy-safe version)
    dx = np.full_like(x[state["diff_window"]:], np.nan)
    dy = np.full_like(y[state["diff_window"]:], np.nan)
    dx = x[state["diff_window"]:] - x[:-state["diff_window"]]
    dy = y[state["diff_window"]:] - y[:-state["diff_window"]]

    # Heading and speed
    heading_direction = np.arctan2(dy, dx)        # radians
    speed = np.hypot(dx, dy)
    heading_direction[speed < speed_threshold] = np.nan

    # Unit vectors from heading angle
    u = np.cos(heading_direction)
    v = np.sin(heading_direction)

    valid = np.isfinite(u) & np.isfinite(v)
    idx = np.where(valid)[0][::state["arrow_every"]]

    # Color by time (0 -> 1 across run)
    t_norm = (idx - idx.min()) / (idx.max() - idx.min() + 1e-12)

    # Plot
    fig, ax = plt.subplots(1,3,figsize=(21, 7))

    Arena(ax = ax[0], condition = homings_dict["condition"][state["idx"]],
                barrier_coordinates = tracking_data["barrier_loc"][:-1], 
                shelter_coordinates=tracking_data["shelter_loc"], full_image = False)

    q = ax[0].quiver(
        x[idx], y[idx],
        u[idx] * state["arrow_len"], v[idx] * state["arrow_len"],
        t_norm,
        cmap="viridis",
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.01,
        headwidth=5,
        headlength=5,
        alpha=0.95
    )

    ax[0].set_title(f"Homing {state['idx']}: heading arrows over trajectory")

    Arena(ax = ax[1], condition = homings_dict["condition"][state["idx"]],
                barrier_coordinates = tracking_data["barrier_loc"][:-1], 
                shelter_coordinates=tracking_data["shelter_loc"], full_image = False)

    # Unit vectors from heading angle
    u = np.cos(tracking_data["hdir"][homings_dict["onset_frames"][state["idx"]]:end])
    v = -np.sin(tracking_data["hdir"][homings_dict["onset_frames"][state["idx"]]:end])

    valid = np.isfinite(u) & np.isfinite(v)
    idx = np.where(valid)[0][::state["arrow_every"]]

    # Color by time (0 -> 1 across run)
    t_norm = (idx - idx.min()) / (idx.max() - idx.min() + 1e-12)

    q = ax[1].quiver(
        x[idx], y[idx],
        u[idx] * state["arrow_len"], v[idx] * state["arrow_len"],
        t_norm,
        cmap="viridis",
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.01,
        headwidth=5,
        headlength=6,
        alpha=0.95
    )

    ax[1].set_title(f"Homing {state['idx']}: hdir arrows over trajectory")

    ax[2].plot(heading_direction, label="Heading Direction (rad)", color='blue')
    ax[2].plot(-tracking_data["hdir"][homings_dict["onset_frames"][state["idx"]]:end], label="Tracking hdir (rad)", color='orange')
    diff = heading_direction + tracking_data["hdir"][homings_dict["onset_frames"][state["idx"]]:end]
    diff = np.arctan2(np.sin(diff), np.cos(diff))  # wraps to [-π, π]
    ax[2].plot(diff, label="Difference (rad)", color='green')
    ax[2].axhline(0, color='black', linestyle='--', linewidth=1)
    ax[2].set_xlabel("Frame")
    ax[2].set_ylabel("radians")
    ax[2].legend()
    ax[2].set_ylim([-np.pi, np.pi])

    plt.tight_layout()
    return fig

viewer.set_plot(plot)
viewer.show()